# [MODEL] 현지 최종 책임

EfficientNet 전이학습 기반 식재료 이미지 분류 학습 노트북의 골격(skeleton)이다.
실제 epoch 학습은 다음 주 데이터셋 확보 후 진행할 예정이며, 지금은 클래스 목록,
train/validation/test 분할 로직, 데이터 증강 설정까지만 정의하고 실행은 보류한다.

In [ ]:
# 노트북 골격 작성 단계이므로 실제 학습(model.fit)은 아직 실행하지 않는다
import os
import random

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. 클래스 목록 확정

외형이 서로 비슷해 혼동되기 쉬운 클래스(예: 양파-마늘, 애호박-오이, 청양고추-오이고추 등)는
제외하고, 형태·색상 구분이 뚜렷한 6개 클래스만 선정한다.

In [ ]:
# 데이터 경로 (실제 데이터셋 확보 후 사용자 환경에 맞게 수정)
RAW_DATA_DIR = './data/raw'
SPLIT_DATA_DIR = './data/split'

# 외형이 뚜렷하게 구분되는 6개 클래스만 선정한다 (5~8개 범위 내)
CLASS_NAMES = ['potato', 'carrot', 'cabbage', 'tomato', 'eggplant', 'paprika']

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print('선정된 클래스:', CLASS_NAMES)

## 2. train / validation / test 8:1:1 분할

클래스별 폴더 구조(`RAW_DATA_DIR/클래스명/*.jpg`)를 가정하고,
각 클래스 내 이미지를 8:1:1 비율로 분리하여 `SPLIT_DATA_DIR` 하위에 복사한다.
실제 데이터셋이 준비되는 다음 주에 호출할 예정이며, 지금은 함수 정의만 해둔다.

In [ ]:
import shutil


def split_dataset(raw_dir, split_dir, class_names, ratios=(0.8, 0.1, 0.1), seed=SEED):
    """클래스별 이미지를 train/val/test 폴더로 8:1:1 비율로 복사한다"""
    rng = random.Random(seed)
    for split in ['train', 'val', 'test']:
        for cls in class_names:
            os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)

    for cls in class_names:
        cls_dir = os.path.join(raw_dir, cls)
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        rng.shuffle(files)

        n_total = len(files)
        n_train = int(n_total * ratios[0])
        n_val = int(n_total * ratios[1])

        splits = {
            'train': files[:n_train],
            'val': files[n_train:n_train + n_val],
            'test': files[n_train + n_val:],
        }

        for split, split_files in splits.items():
            for fname in split_files:
                src = os.path.join(cls_dir, fname)
                dst = os.path.join(split_dir, split, cls, fname)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)

        print(f'{cls}: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')


# TODO: 다음 주 실제 데이터셋 확보 후 아래 호출로 분할을 실행한다
# split_dataset(RAW_DATA_DIR, SPLIT_DATA_DIR, CLASS_NAMES)

## 3. ImageDataGenerator 데이터 증강 설정

회전·좌우 반전·밝기 조절을 적용하는 증강 설정만 정의해두고,
`flow_from_directory` 호출 및 실제 학습은 다음 주로 보류한다.

In [ ]:
# train에만 적용할 증강: 회전, 좌우 반전, 밝기 조절
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    horizontal_flip=True,
    brightness_range=(0.8, 1.2),
)

# validation/test에는 증강을 적용하지 않고 스케일링만 수행한다
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

# TODO: 다음 주 SPLIT_DATA_DIR이 준비되면 아래와 같이 flow_from_directory로 제너레이터를 생성하고
# model.fit으로 실제 epoch 학습을 진행한다
# train_generator = train_datagen.flow_from_directory(
#     os.path.join(SPLIT_DATA_DIR, 'train'), target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASS_NAMES, seed=SEED)
# val_generator = val_test_datagen.flow_from_directory(
#     os.path.join(SPLIT_DATA_DIR, 'val'), target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASS_NAMES, seed=SEED)
# test_generator = val_test_datagen.flow_from_directory(
#     os.path.join(SPLIT_DATA_DIR, 'test'), target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASS_NAMES, shuffle=False)

## 4. 다음 단계 (다음 주 진행 예정)

- 데이터셋 확보 후 `split_dataset` 실행
- `flow_from_directory`로 실제 제너레이터 생성
- EfficientNetB0 백본 로드 및 분류기 헤드 구성
- `model.fit`으로 epoch 학습 진행 및 결과 시각화/평가